# Amazon Inventory Reconciliation Using AI - full pipeline (Colab)
**Aarush Patil - PES1UG24AM005**

Run top to bottom on a free **T4 GPU** (Runtime -> Change runtime type). Total about 1-1.5 h for 30K images. If Colab disconnects, re-run from the cell that stopped: the download resumes from `data/labels.csv`.

## 0. Setup

In [ ]:
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU')

In [ ]:
# Back up outputs to Google Drive (survives disconnects). Set to False to skip.
USE_DRIVE = True
BACKUP = '/content/drive/MyDrive/abid_outputs'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(BACKUP, exist_ok=True)

### Get the code
Upload `amazon-inventory-reconciliation.zip` when prompted (or clone your private repo with a token instead).

In [ ]:
import zipfile
if not os.path.exists('/content/amazon-inventory-reconciliation'):
    from google.colab import files
    up = files.upload()   # choose amazon-inventory-reconciliation.zip
    zipfile.ZipFile(list(up)[0]).extractall('/content')
%cd /content/amazon-inventory-reconciliation
!ls

In [ ]:
!pip -q install scikit-image gradio tqdm

## 1. Data (random 30K-image subset, bins with 0-5 items)

In [ ]:
!python download_data.py --n 30000 --out data --workers 32

In [ ]:
!python build_splits.py --data data

## 2. Classical baselines

In [ ]:
!python baselines.py --data data --out results/baselines.json

## 3. CNN runs (ResNet34: softmax head vs ordinal head)

In [ ]:
!python train_cnn.py --data data --arch resnet34 --head cls --name r34_cls --lr-find --workers 2

In [ ]:
!python train_cnn.py --data data --arch resnet34 --head ordinal --name r34_ord --lr-find --workers 2

In [ ]:
if USE_DRIVE:
    !cp -r runs results {BACKUP}/
    print('backed up to', BACKUP)

## 4. Results (metrics, figures, variance detection)

In [ ]:
!python make_results.py --data data --baselines results/baselines.json --out results
!head -c 2500 results/results.json

## 5. Live demo (Gradio) - open the public link it prints, upload a bin photo from `data/images`

In [ ]:
import json
best = json.load(open('results/results.json'))['best_cnn']
print('best run:', best)
!python demo.py --run runs/{best} --gradio

## 6. Download everything
Download `outputs.zip` and `results.json`.

In [ ]:
!zip -qr outputs.zip results runs/*/metrics.json runs/*/history.csv runs/*/test_preds.csv runs/*/lr_find.json
from google.colab import files
files.download('outputs.zip')
files.download('results/results.json')